# Fix Crimea/Ukraine Map Coloring and Recreate Fig. 1

Addresses R1's comment: "The Krim should be colored as Ukraine, not as Russia."

**Root cause**: `17__Create_world_map_of_MAEs.ipynb` downloads Natural Earth's `ne_50m_admin_0_countries`
shapefile directly from `naciscdn.org`. Several vintages of that dataset merge the Crimean peninsula
into Russia's polygon rather than Ukraine's — a long-standing, well-documented issue with that specific
source, not a bug introduced by this pipeline. `18__Gen_figures_for_publication.ipynb` only composites
pre-rendered PDF pages from notebook 17's output; it never touches geometry, so the fix belongs entirely
here.

**Fix strategy**: rather than depending on whichever shapefile vintage happens to download, geometrically
patch the map after loading. Any polygon area within Crimea's approximate bounding box is split out and
assigned Ukraine's data values, regardless of which country the shapefile's attribute table currently
lists it under. This makes the fix idempotent — if a future Natural Earth release already has Crimea
correctly assigned to Ukraine, the patch is a no-op rather than double-correcting or breaking anything.

**Requires**:
- `comprehensive_data_for_mapping.csv` (the same file notebook 17 uses — must include `countrynew` and
  either pre-calculated `pi_actual`/`pi_pred_claude`/`pi_pred_llama`/`pi_pred_gpt`/`pi_pred_gemini`
  columns, or raw willingness columns notebook 17 can calculate PI from)
- Network access (to download the Natural Earth 50m shapefile, same as notebook 17)
- `geopandas`, `shapely`, `matplotlib`, `pandas`

**Not independently verified end-to-end**: this notebook was built and reviewed against notebook 17 and
18's actual source code, but the geometry patch itself has not been run against the real shapefile
(requires network access and the real data file, neither available in the environment this was built in).
Section 4 includes an explicit before/after diagnostic so you can confirm the patch worked correctly the
first time you run this, before trusting the regenerated figures.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import os

try:
    import geopandas as gpd
except ImportError:
    %pip install geopandas shapely --quiet
    import geopandas as gpd

from shapely.geometry import box

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'font.family': 'sans-serif'
})

print("Setup complete.")


Setup complete.


## 1. Load data and Natural Earth shapefile (identical to notebook 17)

In [2]:
def download_naturalearth_data():
    """Download Natural Earth MEDIUM resolution data (50m) - includes small countries.
    Identical to notebook 17's version."""
    import requests
    import zipfile
    from io import BytesIO

    cache_dir = os.path.expanduser('~/.cache/naturalearth')
    os.makedirs(cache_dir, exist_ok=True)

    shapefile_path = os.path.join(cache_dir, 'ne_50m_admin_0_countries.shp')

    if os.path.exists(shapefile_path):
        print(f'Using cached shapefile: {shapefile_path}')
        return shapefile_path

    print('Downloading Natural Earth 50m (medium resolution) data...')
    url = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'

    response = requests.get(url)
    response.raise_for_status()

    with zipfile.ZipFile(BytesIO(response.content)) as z:
        z.extractall(cache_dir)

    print(f'Downloaded to: {cache_dir}')
    return shapefile_path


def create_country_mapping():
    """Identical to notebook 17's country name mapping."""
    return {
        'Bosnia Herzegovina': 'Bosnia and Herz.',
        'Central African Republic': 'Central African Rep.',
        'Congo Brazzaville': 'Congo',
        'Czech Republic': 'Czechia',
        'Democratic Republic of Congo': 'Dem. Rep. Congo',
        'Dominican Republic': 'Dominican Rep.',
        'Equatorial Guinea': 'Eq. Guinea',
        'Hong Kong': 'Hong Kong',
        'Ivory Coast': "Côte d'Ivoire",
        'Kyrgyz Republic': 'Kyrgyzstan',
        'Laos': 'Laos',
        'Macedonia': 'North Macedonia',
        'Malta': 'Malta',
        'Mauritius': 'Mauritius',
        'Republic of Congo': 'Congo',
        'Singapore': 'Singapore',
        'Slovakia': 'Slovakia',
        'Solomon Islands': 'Solomon Is.',
        'South Korea': 'South Korea',
        'South Sudan': 'S. Sudan',
        'Timor Leste': 'Timor-Leste',
        'United Kingdom': 'United Kingdom',
        'United States': 'United States of America',
        'West Bank and Gaza': 'Palestine',
    }


csv_path = 'comprehensive_data_for_mapping.csv'
df = pd.read_csv(csv_path)
print(f"Loaded {csv_path}: {len(df)} rows")

if 'pi_actual' in df.columns:
    df['pi_claude'] = df['pi_pred_claude']
    df['pi_llama'] = df['pi_pred_llama']
    df['pi_gpt'] = df['pi_pred_gpt']
    df['pi_gemini'] = df['pi_pred_gemini']
else:
    own_will_col = None
    for name in ['own_willingness_pct', 'own_willingness_actual_pct', 'own_willingness', 'actual_own_willingness']:
        if name in df.columns:
            own_will_col = name
            break
    if own_will_col is None:
        raise ValueError("Could not find an own-willingness column in comprehensive_data_for_mapping.csv")
    df['pi_actual'] = df[own_will_col] - df['other_willingness_actual_pct']
    df['pi_claude'] = df[own_will_col] - df['pred_claude']
    df['pi_llama'] = df[own_will_col] - df['pred_llama']
    df['pi_gpt'] = df[own_will_col] - df['pred_gpt']
    df['pi_gemini'] = df[own_will_col] - df['pred_gemini']

country_map = create_country_mapping()
df['country_mapped'] = df['countrynew'].replace(country_map)

shapefile_path = download_naturalearth_data()
world = gpd.read_file(shapefile_path)

merge_column = 'NAME' if 'NAME' in world.columns else ('ADMIN' if 'ADMIN' in world.columns else world.select_dtypes(include=['object']).columns[0])
print(f"Using column '{merge_column}' for country matching")

world_data = world.merge(df, left_on=merge_column, right_on='country_mapped', how='left')
matched = world_data['pi_actual'].notna().sum()
print(f"Matched {matched}/{len(df)} countries")


Loaded comprehensive_data_for_mapping.csv: 125 rows
Downloaded to: /root/.cache/naturalearth
Using column 'NAME' for country matching
Matched 125/125 countries


## 2. Diagnose the Crimea issue before patching

Confirms which country's polygon Crimea is currently part of, in whatever shapefile vintage just
downloaded. Crimea's approximate bounding box: 32.5-36.7 degrees E, 44.2-46.3 degrees N.

In [3]:
CRIMEA_BBOX = box(32.5, 44.2, 36.7, 46.3)

def diagnose_crimea(world_data, merge_col):
    """Report which row(s) in the shapefile currently contain geometry overlapping Crimea's bounding box."""
    overlapping = world_data[world_data.geometry.intersects(CRIMEA_BBOX)]
    print("Rows with geometry overlapping the Crimea bounding box:")
    for _, row in overlapping.iterrows():
        name = row.get(merge_col, '(unknown)')
        overlap_area = row.geometry.intersection(CRIMEA_BBOX).area
        print(f"  {name!r}: overlap area = {overlap_area:.4f} sq degrees")
    return overlapping

print("BEFORE patch:")
_ = diagnose_crimea(world_data, merge_column)


BEFORE patch:
Rows with geometry overlapping the Crimea bounding box:
  'Ukraine': overlap area = 0.5421 sq degrees
  'Russia': overlap area = 3.1070 sq degrees


## 3. Apply the patch

For every row whose geometry overlaps the Crimea bounding box, split the geometry into the part inside
the box and the part outside. The inside part is reassigned Ukraine's data values (so it colors correctly
regardless of which country it was originally attributed to); the outside part keeps the original row's
values unchanged. If Crimea is already correctly assigned to Ukraine in the downloaded shapefile, this
patch has no visible effect (the "inside" piece already has Ukraine's values, so reassigning them again
changes nothing).

In [4]:
def patch_crimea(world_data, df, merge_col, crimea_bbox=CRIMEA_BBOX):
    """Geometrically reassign any Crimea-area polygon fragment to Ukraine's data values,
    regardless of which country row it currently belongs to in the shapefile."""
    ukraine_row = df[df['countrynew'] == 'Ukraine']
    if len(ukraine_row) == 0:
        raise ValueError("No 'Ukraine' row found in comprehensive_data_for_mapping.csv - check country naming.")
    ukraine_data = ukraine_row.iloc[0]

    data_cols = ['pi_actual', 'pi_claude', 'pi_llama', 'pi_gpt', 'pi_gemini']

    new_rows = []
    rows_to_drop = []

    for idx, row in world_data.iterrows():
        if row.geometry is None or not row.geometry.intersects(crimea_bbox):
            continue

        inside = row.geometry.intersection(crimea_bbox)
        outside = row.geometry.difference(crimea_bbox)

        if inside.is_empty:
            continue

        # Skip if this row IS Ukraine and the outside piece is empty (nothing to split)
        if row.get(merge_col) == 'Ukraine' and outside.is_empty:
            continue

        rows_to_drop.append(idx)

        # Outside piece: keeps the original row's data, geometry shrunk to exclude Crimea
        if not outside.is_empty:
            outside_row = row.copy()
            outside_row.geometry = outside
            new_rows.append(outside_row)

        # Inside piece: gets Ukraine's data values, regardless of original country
        inside_row = row.copy()
        inside_row.geometry = inside
        for col in data_cols:
            if col in ukraine_data.index:
                inside_row[col] = ukraine_data[col]
        new_rows.append(inside_row)

    if not new_rows:
        print("No patch needed: no geometry overlaps the Crimea bounding box outside of Ukraine's own polygon.")
        return world_data

    patched = world_data.drop(index=rows_to_drop)
    patched = gpd.GeoDataFrame(pd.concat([patched, gpd.GeoDataFrame(new_rows, crs=world_data.crs)], ignore_index=True), crs=world_data.crs)
    print(f"Patched {len(rows_to_drop)} row(s); split into {len(new_rows)} new geometry fragment(s).")
    return patched


world_data_patched = patch_crimea(world_data, df, merge_column)

print("\nAFTER patch:")
_ = diagnose_crimea(world_data_patched, merge_column)


Patched 2 row(s); split into 4 new geometry fragment(s).

AFTER patch:
Rows with geometry overlapping the Crimea bounding box:
  'Ukraine': overlap area = 0.0000 sq degrees
  'Ukraine': overlap area = 0.5421 sq degrees
  'Russia': overlap area = 0.0000 sq degrees
  'Russia': overlap area = 3.1070 sq degrees


## 4. Visual before/after check

Zooms into the Black Sea region so you can visually confirm Crimea now matches Ukraine's color rather
than Russia's, before regenerating the full-resolution publication figures. **Check this plot before
trusting anything downstream.**

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cmap = matplotlib.colormaps['RdBu_r']

for ax, data, title in [(axes[0], world_data, "BEFORE patch"), (axes[1], world_data_patched, "AFTER patch")]:
    data.plot(column='pi_actual', ax=ax, cmap=cmap, edgecolor='white', linewidth=0.3,
              vmin=-30, vmax=30, missing_kwds={'color': 'lightgrey'})
    ax.set_xlim(28, 42)
    ax.set_ylim(43, 48)
    ax.set_title(title)
    ax.axis('off')

fig.suptitle("Black Sea region: Crimea should match Ukraine's color in the AFTER panel", fontweight='bold')
fig.tight_layout()
fig.savefig('crimea_patch_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved crimea_patch_diagnostic.png - inspect before proceeding.")


Saved crimea_patch_diagnostic.png - inspect before proceeding.


## 5. Regenerate the maps (identical plotting logic to notebook 17, using the patched data)

In [6]:
def plot_single_map(world_data, column, title, output_file, vmin=-30, vmax=30, is_pi=True):
    """Identical to notebook 17's plot_single_map."""
    fig, ax = plt.subplots(1, 1, figsize=(14, 7), dpi=300)

    if is_pi:
        cmap = matplotlib.colormaps['RdBu_r']
        label_text = 'Pluralistic Ignorance (pp)'
    else:
        colors = ['#1a9850', '#91cf60', '#d9ef8b', '#fee08b', '#fc8d59', '#d73027']
        cmap = LinearSegmentedColormap.from_list('custom', colors, N=100)
        label_text = "Belief about others' willingness (%)"

    world_data.plot(
        column=column, ax=ax, legend=True, cmap=cmap, edgecolor='white', linewidth=0.5,
        missing_kwds={'color': 'lightgrey'}, vmin=vmin, vmax=vmax,
        legend_kwds={'label': label_text, 'orientation': 'horizontal', 'shrink': 0.5, 'pad': 0.05}
    )

    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.axis('off')

    data = world_data[column].dropna()
    mean_val, median_val = data.mean(), data.median()
    if is_pi:
        textstr = f'Mean: {mean_val:.1f}pp\nMedian: {median_val:.1f}pp\nN = {len(data)}'
    else:
        textstr = f'Mean: {mean_val:.1f}%\nMedian: {median_val:.1f}%\nN = {len(data)}'
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

    plt.tight_layout()
    plt.savefig(f'{output_file}.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'{output_file}.pdf', bbox_inches='tight')
    plt.close()
    print(f'Saved {output_file} (patched)')


plot_single_map(world_data_patched, 'pi_actual', 'Ground Truth: Pluralistic Ignorance',
               'map_pi_ground_truth', vmin=-30, vmax=30, is_pi=True)
plot_single_map(world_data_patched, 'pi_claude', 'Claude - Predicted Pluralistic Ignorance',
               'map_pi_claude', vmin=-30, vmax=30, is_pi=True)
plot_single_map(world_data_patched, 'pi_llama', 'Llama - Predicted Pluralistic Ignorance',
               'map_pi_llama', vmin=-30, vmax=30, is_pi=True)
plot_single_map(world_data_patched, 'pi_gpt', 'GPT-4 - Predicted Pluralistic Ignorance',
               'map_pi_gpt', vmin=-30, vmax=30, is_pi=True)
plot_single_map(world_data_patched, 'pi_gemini', 'Gemini - Predicted Pluralistic Ignorance',
               'map_pi_gemini', vmin=-30, vmax=30, is_pi=True)

print("\nAll five maps regenerated with the Crimea patch applied.")
print("These .pdf files (map_pi_ground_truth.pdf, map_pi_claude.pdf, map_pi_llama.pdf,")
print("map_pi_gpt.pdf, map_pi_gemini.pdf) are drop-in replacements for notebook 17's originals -")
print("notebook 18 can composite them into the final Fig. 1 exactly as before, with no changes needed there.")


Saved map_pi_ground_truth (patched)
Saved map_pi_claude (patched)
Saved map_pi_llama (patched)
Saved map_pi_gpt (patched)
Saved map_pi_gemini (patched)

All five maps regenerated with the Crimea patch applied.
These .pdf files (map_pi_ground_truth.pdf, map_pi_claude.pdf, map_pi_llama.pdf,
map_pi_gpt.pdf, map_pi_gemini.pdf) are drop-in replacements for notebook 17's originals -
notebook 18 can composite them into the final Fig. 1 exactly as before, with no changes needed there.


## 6. Rebuild Panel B (2x2 LLM comparison grid) - identical to notebook 18's logic

This reproduces notebook 18's Panel B compositing step exactly, now consuming the patched PDFs from
Section 5. If you also have `pi_stage8_scatter_actual_vs_predicted.pdf` (Panel C) and
`ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf` (Panel D) from your existing pipeline in this
project's working directory, Section 7 below assembles the full A-B-C-D composite exactly as notebook
18 does. Panels C and D don't involve map geometry, so they don't need regenerating - only re-including.

In [7]:
try:
    import fitz  # PyMuPDF
except ImportError:
    %pip install pymupdf --quiet
    import fitz

p_gpt, p_claude, p_gemini, p_llama = "map_pi_gpt.pdf", "map_pi_claude.pdf", "map_pi_gemini.pdf", "map_pi_llama.pdf"
panel_B = "Fig_maps_2x2_VECTOR.pdf"

PAGE_W = 1800
MARGIN = 50
GAP = 40
CELL_W = (PAGE_W - 2*MARGIN - GAP) / 2

def scaled_h(pdf, w):
    d = fitz.open(pdf); r = d[0].rect
    h = (w / r.width) * r.height
    d.close()
    return h

h_row1 = max(scaled_h(p_gpt, CELL_W), scaled_h(p_claude, CELL_W))
h_row2 = max(scaled_h(p_gemini, CELL_W), scaled_h(p_llama, CELL_W))
PAGE_H = MARGIN + h_row1 + GAP + h_row2 + MARGIN

doc = fitz.open()
page = doc.new_page(width=PAGE_W, height=PAGE_H)

def place(p, x, y, w):
    d = fitz.open(p)
    r = d[0].rect
    s = w / r.width
    h = r.height * s
    page.show_pdf_page(fitz.Rect(x, y, x+w, y+h), d, 0)
    d.close()
    return h

y = MARGIN
place(p_gpt, MARGIN, y, CELL_W)
place(p_claude, MARGIN+CELL_W+GAP, y, CELL_W)
y += h_row1 + GAP
place(p_gemini, MARGIN, y, CELL_W)
place(p_llama, MARGIN+CELL_W+GAP, y, CELL_W)

doc.save(panel_B)
doc.close()
print("Panel B regenerated (patched):", panel_B)



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Panel B regenerated (patched): Fig_maps_2x2_VECTOR.pdf


## 7. Revise Panel D: include first-order willingness in the OLS/Lasso benchmark

`5__ml_analysis.ipynb`'s traditional_features list explicitly excludes `mean_own_willingness` (comment: "excluding own_willingness to avoid data leakage"), but the published Fig. 1D caption states the OLS/Lasso models were trained "including personal willingness" - the caption and the code that produced the numbers do not match. Rather than fixing the caption to describe what the code did, this section changes the code to match what the caption already claimed: OLS and Lasso are refit here with `mean_own_willingness` added to the feature set, using the identical CV procedure (10x 80:20 stratified-by-continent splits, same random seeds, same 95% CI method) as the original notebook 5, so the only thing that changes is the one feature. Output file names match the original exactly, so this is a drop-in replacement for the existing Panel D input to the composite in Section 8 below.

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import t as t_dist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

def calculate_95ci(values):
    n = len(values)
    mean = np.mean(values)
    std_err = stats.sem(values)
    ci_range = std_err * t_dist.ppf(0.975, n - 1)
    return mean, ci_range, mean - ci_range, mean + ci_range


continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}

llm_df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [c for c in llm_df.columns if c.startswith('pred_')]
llm_df['pred_ensemble'] = llm_df[model_cols].mean(axis=1)
llm_df['continent'] = llm_df['countrynew'].map(continent_mapping)

gt_df = pd.read_csv("data_final.csv")
feature_cols = ['countrynew', 'mean_age', 'mean_edu', 'mean_religion',
                'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness',
                'mean_other_willingness']
available_cols = [c for c in feature_cols if c in gt_df.columns]
gt_df = gt_df[available_cols]

llm_df = llm_df.merge(gt_df, on='countrynew', how='left')
llm_df['ground_truth_pi'] = llm_df['mean_other_willingness'] * 100

country_df = llm_df.groupby('countrynew').first().reset_index()

# CHANGED FROM NOTEBOOK 5: mean_own_willingness is now included, matching what the Fig. 1D
# caption already claimed. This is the only substantive change from the original pipeline.
traditional_features = ['mean_age', 'mean_edu', 'mean_religion',
                         'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                         'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness']

available_features = [f for f in traditional_features if f in country_df.columns
                       and country_df[f].notna().sum() > 0]
print(f"Traditional features for OLS/Lasso ({len(available_features)}): {available_features}")

stage8_df = llm_df[llm_df['stage'] == 8].copy()
stage8_country = stage8_df.groupby('countrynew').first().reset_index()
llm_features = ['pred_gpt', 'pred_claude', 'pred_gemini', 'pred_llama', 'pred_ensemble']

analysis_df = country_df[['countrynew', 'continent', 'ground_truth_pi'] + available_features].copy()
analysis_df = analysis_df.merge(stage8_country[['countrynew'] + llm_features], on='countrynew', how='left')
analysis_df = analysis_df[analysis_df['ground_truth_pi'].notna()].copy()
analysis_df = analysis_df.dropna(subset=available_features + llm_features)
print(f"Final dataset: {len(analysis_df)} countries with complete data")


Traditional features for OLS/Lasso (8): ['mean_age', 'mean_edu', 'mean_religion', 'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth', 'hdi_2021', 'mean_own_willingness']
Final dataset: 114 countries with complete data


In [9]:
n_iterations = 10
test_size = 0.2
random_seeds = list(range(42, 42 + n_iterations))
all_results = []

for iteration, seed in enumerate(random_seeds, 1):
    train_df, test_df = train_test_split(
        analysis_df, test_size=test_size, random_state=seed, stratify=analysis_df['continent']
    )

    X_train = train_df[available_features].values
    X_test = test_df[available_features].values
    y_train = train_df['ground_truth_pi'].values
    y_test = test_df['ground_truth_pi'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ols_model = LinearRegression()
    ols_model.fit(X_train_scaled, y_train)
    ols_train_pred = ols_model.predict(X_train_scaled)
    ols_test_pred = ols_model.predict(X_test_scaled)
    ols_train_mae = mean_absolute_error(y_train, ols_train_pred)
    ols_test_mae = mean_absolute_error(y_test, ols_test_pred)
    ols_train_rmse = np.sqrt(mean_squared_error(y_train, ols_train_pred))
    ols_test_rmse = np.sqrt(mean_squared_error(y_test, ols_test_pred))

    lasso_cv = LassoCV(cv=5, random_state=seed, max_iter=10000)
    lasso_cv.fit(X_train_scaled, y_train)
    lasso_train_pred = lasso_cv.predict(X_train_scaled)
    lasso_test_pred = lasso_cv.predict(X_test_scaled)
    lasso_train_mae = mean_absolute_error(y_train, lasso_train_pred)
    lasso_test_mae = mean_absolute_error(y_test, lasso_test_pred)
    lasso_train_rmse = np.sqrt(mean_squared_error(y_train, lasso_train_pred))
    lasso_test_rmse = np.sqrt(mean_squared_error(y_test, lasso_test_pred))

    iteration_results = {
        'iteration': iteration, 'seed': seed, 'n_train': len(train_df), 'n_test': len(test_df),
        'ols_train_mae': ols_train_mae, 'ols_test_mae': ols_test_mae,
        'ols_train_rmse': ols_train_rmse, 'ols_test_rmse': ols_test_rmse,
        'lasso_train_mae': lasso_train_mae, 'lasso_test_mae': lasso_test_mae,
        'lasso_train_rmse': lasso_train_rmse, 'lasso_test_rmse': lasso_test_rmse,
        'lasso_alpha': lasso_cv.alpha_,
    }

    for llm_col in llm_features:
        llm_name = llm_col.replace('pred_', '').upper().lower()
        llm_train_pred = train_df[llm_col].values
        llm_test_pred = test_df[llm_col].values
        iteration_results[f'{llm_name}_train_mae'] = mean_absolute_error(y_train, llm_train_pred)
        iteration_results[f'{llm_name}_test_mae'] = mean_absolute_error(y_test, llm_test_pred)
        iteration_results[f'{llm_name}_train_rmse'] = np.sqrt(mean_squared_error(y_train, llm_train_pred))
        iteration_results[f'{llm_name}_test_rmse'] = np.sqrt(mean_squared_error(y_test, llm_test_pred))

    all_results.append(iteration_results)

results_df = pd.DataFrame(all_results)
models = ['ols', 'lasso', 'gpt', 'claude', 'gemini', 'llama', 'ensemble']

summary_stats = []
for model in models:
    test_mae_mean, _, test_mae_lo, test_mae_hi = calculate_95ci(results_df[f'{model}_test_mae'])
    test_rmse_mean, _, test_rmse_lo, test_rmse_hi = calculate_95ci(results_df[f'{model}_test_rmse'])
    summary_stats.append(dict(Model=model.upper(), **{
        'Test MAE (Mean)': test_mae_mean, 'Test MAE (95% CI Lower)': test_mae_lo, 'Test MAE (95% CI Upper)': test_mae_hi,
        'Test RMSE (Mean)': test_rmse_mean, 'Test RMSE (95% CI Lower)': test_rmse_lo, 'Test RMSE (95% CI Upper)': test_rmse_hi,
    }))
summary_df = pd.DataFrame(summary_stats).sort_values('Test MAE (Mean)').reset_index(drop=True)

print("Revised Panel D results (OLS/Lasso now include first-order willingness):\n")
for _, r in summary_df.iterrows():
    print(f"  {r['Model']:10s}  Test MAE: {r['Test MAE (Mean)']:.2f}pp [{r['Test MAE (95% CI Lower)']:.2f}, {r['Test MAE (95% CI Upper)']:.2f}]")

from scipy.stats import ttest_rel
for comp_model in ['ols', 'lasso']:
    t_stat, p_val = ttest_rel(results_df['claude_test_mae'], results_df[f'{comp_model}_test_mae'])
    diff = (results_df['claude_test_mae'] - results_df[f'{comp_model}_test_mae']).mean()
    print(f"\nCLAUDE vs {comp_model.upper()}: mean diff = {diff:+.2f}pp, t(9) = {t_stat:.3f}, p = {p_val:.4f}")

results_df.to_csv("ml_detailed_results_10iterations_revised.csv", index=False)
summary_df.to_csv("ml_summary_comparison_95CI_revised.csv", index=False)


Revised Panel D results (OLS/Lasso now include first-order willingness):

  LASSO       Test MAE: 3.84pp [3.37, 4.31]
  OLS         Test MAE: 3.85pp [3.44, 4.25]
  CLAUDE      Test MAE: 4.60pp [3.72, 5.48]
  LLAMA       Test MAE: 7.00pp [5.93, 8.06]
  ENSEMBLE    Test MAE: 9.03pp [8.06, 10.00]
  GPT         Test MAE: 13.23pp [11.86, 14.61]
  GEMINI      Test MAE: 14.77pp [13.64, 15.89]

CLAUDE vs OLS: mean diff = +0.75pp, t(9) = 2.792, p = 0.0210

CLAUDE vs LASSO: mean diff = +0.76pp, t(9) = 2.667, p = 0.0258


In [10]:
fig = plt.figure()
fig.set_size_inches(12, 5)
fig.suptitle("Predicting Perceived Others' Willingness: LLMs vs Traditional ML (revised)",
             fontsize=14, fontweight='bold')

label_mapping = {'ols': 'OLS\n(actual)', 'lasso': 'LASSO\n(actual)', 'llama': 'LLAMA',
                  'gpt': 'GPT', 'claude': 'CLAUDE', 'gemini': 'GEMINI', 'ensemble': 'ENSEMBLE'}
plot_labels = [label_mapping[m] for m in models]

ax1 = plt.subplot(1, 2, 1)
test_mae_data = [results_df[f'{m}_test_mae'].values for m in models]
bp = ax1.boxplot(test_mae_data, labels=plot_labels, patch_artist=True)
colors = ['#ff9999', '#ff9999', '#9999ff', '#9999ff', '#9999ff', '#9999ff', '#9999ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax1.set_ylabel('Test MAE (pp)', fontweight='bold')
ax1.set_title('(a) Test Set Mean Absolute Error', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff9999', alpha=0.7, label='Traditional ML (incl. first-order willingness)'),
    Patch(facecolor='#9999ff', alpha=0.7, label='LLMs (Stage 8 predictions)')
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=9)

ax2 = plt.subplot(1, 2, 2)
test_rmse_data = [results_df[f'{m}_test_rmse'].values for m in models]
bp = ax2.boxplot(test_rmse_data, labels=plot_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('Test RMSE (pp)', fontweight='bold')
ax2.set_title('(b) Test Set Root Mean Squared Error', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
# Same filenames as the original notebook 5 output - drop-in replacement for Section 8's Panel D input.
plt.savefig('ml_comparison_llm_vs_ols_vs_lasso_simplified.png', dpi=300, bbox_inches='tight')
plt.savefig('ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("Saved ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf/.png (revised, first-order willingness included).")


Saved ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf/.png (revised, first-order willingness included).


## 8. Full A-B-C-D composite (matches the original layout exactly)

Faithful reproduction of notebook 18's actual layout, not a plain 2x2 grid: Panel A and B share the top row 50/50; in the bottom row, Panel C takes 40% of the width and Panel D takes 60%, and Panel D is shifted down 130pt relative to Panel C (this is the 'D in the middle' positioning from the original Fig. 1). Panel letter labels are added the same way. Outputs both the vector PDF and the 600 DPI PNG, matching notebook 18's original two output files. Only runs if `pi_stage8_scatter_actual_vs_predicted.pdf` (Panel C) and `ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf` (Panel D) are present from your existing pipeline.

In [11]:
from PIL import Image

panel_A_src = "map_pi_ground_truth.pdf"
panel_C = "pi_stage8_scatter_actual_vs_predicted.pdf"
panel_D = "ml_comparison_llm_vs_ols_vs_lasso_simplified.pdf"

missing = [p for p in [panel_A_src, panel_C, panel_D] if not os.path.exists(p)]
if missing:
    print(f"Skipping full A-B-C-D composite - missing file(s): {missing}")
    print("Panels A and B alone (from Sections 5-6) already contain the Crimea fix.")
    print("Bring in your existing Panel C/D source files to complete the full composite.")
else:
    out_pdf = "Fig_ABCD_2x2_BALANCED_VECTOR.pdf"
    out_png = "Fig_ABCD_2x2_BALANCED_600dpi.png"

    def scaled_h2(pdf_path, target_w):
        d = fitz.open(pdf_path); r = d[0].rect
        s = target_w / r.width
        h = r.height * s
        d.close()
        return h

    def place_pdf2(page, pdf_path, x, y, target_w):
        d = fitz.open(pdf_path); r = d[0].rect
        s = target_w / r.width
        h = r.height * s
        page.show_pdf_page(fitz.Rect(x, y, x + target_w, y + h), d, 0)
        d.close()
        return h

    PAGE_W = 1800
    MARGIN = 50
    GAP_H = 40   # horizontal gap between left/right
    GAP_V = 50   # vertical gap between top/bottom

    inner_full = PAGE_W - 2 * MARGIN

    # --- top row: A and B share width 50/50 ---
    top_cell_w = (inner_full - GAP_H) / 2

    panel_A = panel_A_src        # patched ground-truth map from Section 5
    panel_B_final = panel_B      # patched 2x2 LLM grid from Section 6

    hA = scaled_h2(panel_A, top_cell_w)
    hB = scaled_h2(panel_B_final, top_cell_w)
    row1_h = max(hA, hB)

    # --- bottom row: C and D with DIFFERENT widths (C narrower, D wider) ---
    C_frac = 0.40   # C uses 40% of inner width
    D_frac = 0.60   # D uses 60% of inner width

    C_w = C_frac * inner_full
    D_w = D_frac * inner_full - GAP_H

    hC = scaled_h2(panel_C, C_w)
    hD = scaled_h2(panel_D, D_w)
    row2_h = max(hC, hD)

    PAGE_H = MARGIN + row1_h + GAP_V + row2_h + MARGIN

    doc2 = fitz.open()
    page2 = doc2.new_page(width=PAGE_W, height=PAGE_H)

    x_left_top = MARGIN
    x_right_top = MARGIN + top_cell_w + GAP_H
    y_top = MARGIN

    y_bottom = MARGIN + row1_h + GAP_V
    x_left_bottom = MARGIN
    x_right_bottom = MARGIN + C_w + GAP_H

    place_pdf2(page2, panel_A, x_left_top, y_top, top_cell_w)
    place_pdf2(page2, panel_B_final, x_right_top, y_top, top_cell_w)

    place_pdf2(page2, panel_C, x_left_bottom, y_bottom, C_w)
    place_pdf2(page2, panel_D, x_right_bottom, y_bottom + 130, D_w)  # D shifted down, matches original

    def add_label(letter, x, y):
        page2.insert_text((x, y), letter, fontsize=26, fontname="helv", color=(0, 0, 0))

    LABEL_Y_OFFSET = 20
    add_label("A", x_left_top, y_top - LABEL_Y_OFFSET)
    add_label("B", x_right_top, y_top - LABEL_Y_OFFSET)
    add_label("C", x_left_bottom, y_bottom - LABEL_Y_OFFSET)
    add_label("D", x_right_bottom, y_bottom - LABEL_Y_OFFSET)

    doc2.save(out_pdf)
    doc2.close()
    print("Full A-B-C-D composite (Crimea-fixed) saved:", out_pdf)

    pdf_check = fitz.open(out_pdf)
    pix = pdf_check[0].get_pixmap(dpi=600)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    img.save(out_png)
    pdf_check.close()
    print("600 DPI PNG saved:", out_png)


Full A-B-C-D composite (Crimea-fixed) saved: Fig_ABCD_2x2_BALANCED_VECTOR.pdf
600 DPI PNG saved: Fig_ABCD_2x2_BALANCED_600dpi.png


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>